# Step 1: Exact and Fuzzy Matching

This is the first prediction step of our cascade pipeline. The idea is to handle the easy cases before reaching for more complex methods.

**Exact match:** If the same literal appeared in training, we already know its code. This gives us high-confidence predictions for a large fraction of the test set.

**Fuzzy match:** For literals that don't have an exact match, we look for the closest training literal using Levenshtein distance (edit distance). This catches typos, minor spelling variations, and abbreviations that were written slightly differently.

Files to upload to Colab before running:
- `train_preprocessed.csv` (from Step 0)
- `test_preprocessed.csv` (from Step 0)

Files produced at the end:
- `step1_predictions.csv` (test predictions with the method used for each one)

## 1. Install and import

`rapidfuzz` is a fast Python library for fuzzy string matching. It implements Levenshtein distance and similar algorithms in C++ so it's much faster than pure Python implementations.

In [1]:
!pip install rapidfuzz -q


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
from rapidfuzz import fuzz, process
import time

## 2. Load the preprocessed data

In [3]:
train = pd.read_csv('train_preprocessed.csv')
test = pd.read_csv('test_preprocessed.csv')

print(f'Training: {len(train):,} rows')
print(f'Test:     {len(test):,} rows')
print(f'\nTraining columns: {list(train.columns)}')
print(f'Test columns:     {list(test.columns)}')

Training: 13,700 rows
Test:     6,667 rows

Training columns: ['Code', 'Literal', 'Literal_clean', 'Literal_norm', 'code_type', 'chapter', 'is_icd10', 'D_P', 'Description']
Test columns:     ['id', 'Literal', 'Literal_clean', 'Literal_norm']


## 3. Build the exact match lookup table

For each unique normalized literal in training, we record the most common code. This handles the 19% of literals that are ambiguous (same text, different codes in different rows). The most-common-code strategy is the safest single answer we can give.

We use the `Literal_norm` column from Step 0 (lowercase, no accents) so that minor formatting differences don't break the match.

In [4]:
exact_lookup = {}

for lit_norm, group in train.groupby('Literal_norm'):
    # value_counts returns codes sorted by frequency, most common first
    most_common_code = group['Code'].value_counts().index[0]
    exact_lookup[lit_norm] = most_common_code

print(f'Unique normalized literals in training: {len(exact_lookup):,}')
print(f'\nExamples from the lookup table:')
for lit, code in list(exact_lookup.items())[:8]:
    print(f'  "{lit}" -> {code}')

Unique normalized literals in training: 8,355

Examples from the lookup table:
  ", color: tenido" -> O770
  ", ex fumador" -> Z87891
  ", se extrae coagulo" -> 10D17Z9
  "- grupo sanguineo: a positivo" -> Z6710
  "- rx torax y abdomen" -> BW03ZZZ
  ". a negativo" -> Z6711
  ". a positivo" -> Z6710
  ". desgarro i" -> O700


## 4. Apply exact matching to the test set

In [5]:
# .map() returns NaN for keys not in the dictionary
test['exact_code'] = test['Literal_norm'].map(exact_lookup)

n_exact = test['exact_code'].notna().sum()
n_remaining = test['exact_code'].isna().sum()

print(f'Exact matches:    {n_exact:,} ({n_exact/len(test)*100:.1f}% of test)')
print(f'Need fuzzy match: {n_remaining:,} ({n_remaining/len(test)*100:.1f}% of test)')

Exact matches:    3,812 (57.2% of test)
Need fuzzy match: 2,855 (42.8% of test)


In [6]:
# A look at some exact matches to verify they look sensible
test[test['exact_code'].notna()].sample(8, random_state=1)[['Literal', 'Literal_norm', 'exact_code']]

,Literal,Literal_norm,exact_code
588,obesidad part,obesidad part,O99214
3778,TAC craneal,tac craneal,B020ZZZ
2622,Gemelar anteparto,gemelar anteparto,65103
3116,Talla baja,talla baja,78343
2472,pielonefritis de repetición,pielonefritis de repeticion,Z87440
2205,Esofagitis eosinófila,esofagitis eosinofila,53013
4712,Mastectomía izquierda,mastectomia izquierda,0HTU0ZZ
703,carcinoma lobulillar infiltrante,carcinoma lobulillar infiltrante,85223


In [7]:
# A look at what failed exact match - these are the candidates for fuzzy matching
test[test['exact_code'].isna()].sample(15, random_state=2)[['Literal', 'Literal_norm']]

,Literal,Literal_norm
2144,- D.Mellitus,- d.mellitus
341,LUPUS,lupus
373,polipectomia ciego,polipectomia ciego
3578,angina crónica estable,angina cronica estable
6498,K40.00,k40.00
6659,Espondiloartrosis lumbar,espondiloartrosis lumbar
6635,antrostomía maxilar izquierda,antrostomia maxilar izquierda
2312,historia muerte neonato,historia muerte neonato
2649,Colitis ulcerosa pancolitis,colitis ulcerosa pancolitis
6656,Osteroporosis,osteroporosis


## 5. Quick demonstration of fuzzy matching

Before running fuzzy matching on the full test set, let's see how the scoring behaves on some examples. `fuzz.ratio` returns a similarity score from 0 (totally different) to 100 (identical).

In [8]:
demo_pairs = [
    ('absceso mama', 'abceso mama'),         # one letter typo
    ('hipotiroidismo', 'hipotoridismo'),       # transposed letters
    ('hernia umbilical', 'hernia umbilical'),  # identical
    ('hernia umbilical', 'hernia inguinal'),   # similar structure, different meaning
    ('shock septico', 'diabetes tipo 2'),      # totally unrelated
]

print(f'{"String A":<25} {"String B":<25} {"Score":>6}')
print('-' * 60)
for a, b in demo_pairs:
    score = fuzz.ratio(a, b)
    print(f'{a:<25} {b:<25} {score:>6.1f}')

String A                  String B                   Score
------------------------------------------------------------
absceso mama              abceso mama                 95.7
hipotiroidismo            hipotoridismo               88.9
hernia umbilical          hernia umbilical           100.0
hernia umbilical          hernia inguinal             71.0
shock septico             diabetes tipo 2             35.7


Notice how the score gracefully drops as the strings become less similar. Near-identical strings score 90+, similar-but-different score around 70-80, and unrelated strings score around 30 or lower.

## 6. Run fuzzy matching on the unresolved test literals

This is the slow part. For each unmatched test literal, we compare it against every unique training literal and keep the best match. This takes 2-3 minutes.

In [9]:
# Get the list of all unique training literals
train_literals = list(exact_lookup.keys())

# Get the test literals that did NOT get an exact match
unmatched = test[test['exact_code'].isna()].copy()

print(f'Running fuzzy matching on {len(unmatched):,} unmatched test literals')
print(f'Searching against {len(train_literals):,} unique training literals')
print('This will take 2-3 minutes...')

start = time.time()

# Store results in a list, convert to dataframe at the end
fuzzy_results = []

for idx, row in unmatched.iterrows():
    # process.extractOne returns (best_match_string, score, index_in_choices)
    # score_cutoff=0 means return the best match even if it's a poor one
    result = process.extractOne(
        row['Literal_norm'],
        train_literals,
        scorer=fuzz.ratio,
        score_cutoff=0
    )
    if result is not None:
        best_literal, score, _ = result
        fuzzy_results.append({
            'test_idx': idx,
            'test_literal': row['Literal_norm'],
            'matched_literal': best_literal,
            'matched_code': exact_lookup[best_literal],
            'fuzzy_score': score
        })

fuzzy_df = pd.DataFrame(fuzzy_results)
elapsed = time.time() - start
print(f'\nDone in {elapsed:.1f} seconds')
print(f'Got fuzzy results for {len(fuzzy_df):,} literals')

Running fuzzy matching on 2,855 unmatched test literals
Searching against 8,355 unique training literals
This will take 2-3 minutes...

Done in 2.3 seconds
Got fuzzy results for 2,855 literals


## 7. Examine the fuzzy match score distribution

Not every fuzzy match is trustworthy. We need to pick a threshold above which we accept the match. Let's first see how the scores are distributed.

In [10]:
print('Score distribution:')
print(fuzzy_df['fuzzy_score'].describe())

print('\nScores by bucket:')
for lo, hi in [(0,50), (50,60), (60,70), (70,80), (80,85), (85,90), (90,95), (95,100), (100,101)]:
    n = ((fuzzy_df.fuzzy_score >= lo) & (fuzzy_df.fuzzy_score < hi)).sum()
    print(f'  {lo:3d} - {hi:3d}: {n:5,} literals')

Score distribution:
count    2855.000000
mean       83.845501
std        10.918901
min        50.000000
25%        75.675676
50%        85.714286
75%        93.617021
max        98.924731
Name: fuzzy_score, dtype: float64

Scores by bucket:
    0 -  50:     0 literals
   50 -  60:    36 literals
   60 -  70:   338 literals
   70 -  80:   568 literals
   80 -  85:   444 literals
   85 -  90:   387 literals
   90 -  95:   525 literals
   95 - 100:   557 literals
  100 - 101:     0 literals


In [11]:
# High-confidence fuzzy matches (score >= 85) - look correct on inspection
print('Sample HIGH confidence fuzzy matches:')
high = fuzzy_df[fuzzy_df.fuzzy_score >= 85].sort_values('fuzzy_score', ascending=False)
for _, row in high.head(12).iterrows():
    print(f'  [{row.fuzzy_score:5.1f}] "{row.test_literal}" -> "{row.matched_literal}" ({row.matched_code})')

Sample HIGH confidence fuzzy matches:
  [ 98.9] "insuficienciarespiratoria hipercapnica cronica" -> "insuficiencia respiratoria hipercapnica cronica" (51883)
  [ 98.8] "taquicardia supraventricular paroxisitica" -> "taquicardia supraventricular paroxistica" (I471)
  [ 98.7] "trastorno esquizoafectivo, tipo bipolar" -> "trastorno esquizoafectivo tipo bipolar" (F250)
  [ 98.7] "grupo sanguineo materno: o rh: positivo" -> "grupo sanguineo materno: o rh positivo" (Z6740)
  [ 98.7] "grupo sanguineo materno: a rh: positivo" -> "grupo sanguineo materno: a rh positivo" (Z6710)
  [ 98.7] "grupo sanguineo materno:0 rh positivo" -> "grupo sanguineo materno: 0 rh positivo" (Z6740)
  [ 98.6] "ateromatosis calcificada aortoiliaca" -> "ateromatosis calcificada aortoiliaca." (I700)
  [ 98.6] "ateromatosis calcificada aortoiliaca" -> "ateromatosis calcificada aortoiliaca." (I700)
  [ 98.6] "artropatia degenerativa generalizada" -> "artropatia degenerativa generalitzada" (71509)
  [ 98.6] "grupo sanguin

In [12]:
# Medium confidence (70-85) - some right, some wrong
print('Sample MEDIUM confidence fuzzy matches:')
med = fuzzy_df[(fuzzy_df.fuzzy_score >= 70) & (fuzzy_df.fuzzy_score < 85)]
for _, row in med.sample(min(12, len(med)), random_state=3).iterrows():
    print(f'  [{row.fuzzy_score:5.1f}] "{row.test_literal}" -> "{row.matched_literal}" ({row.matched_code})')

Sample MEDIUM confidence fuzzy matches:
  [ 81.8] "polipectomia endometrial histeroscopica" -> "polipectomia histeroscopica" (0UB98ZZ)
  [ 75.7] "enfermedad behcet" -> "enfermedad de berger" (N028)
  [ 84.8] "ecografia uter" -> "ecografia tv: utero" (BU4CZZZ)
  [ 74.4] "nodulo pulmonar solitario" -> "nodulos pulmonares" (51889)
  [ 70.6] "accidente laboral" -> "accidente secuela" (X58XXXS)
  [ 78.3] "alergica a dexketoprofeno" -> "alergica a ibuprofeno" (V146)
  [ 84.6] "parto gemelos" -> "parto gemelar" (65101)
  [ 72.7] "alergias: cefuroxima" -> "alergias: cefalosporinas" (Z881)
  [ 72.3] "biopsia de arteria temporal" -> "biopsia de peritoneo" (5423)
  [ 83.7] "estat de gastrostomia" -> "estat de traqueostomia" (Z930)
  [ 81.6] "protesis mecanica mitral" -> "protesis mecanica aortica" (Z952)
  [ 75.9] "lipoma en antebrazo izquierdo" -> "quemadura antebrazo izquierdo" (0JBH0ZZ)


In [13]:
# Low confidence (<70) - usually wrong
print('Sample LOW confidence fuzzy matches:')
low = fuzzy_df[fuzzy_df.fuzzy_score < 70]
for _, row in low.sample(min(12, len(low)), random_state=4).iterrows():
    print(f'  [{row.fuzzy_score:5.1f}] "{row.test_literal}" -> "{row.matched_literal}" ({row.matched_code})')

Sample LOW confidence fuzzy matches:
  [ 61.5] "down" -> "sd . down" (7580)
  [ 60.0] "antecedentes familiares de miocardiopatia" -> "antecedente de cancer de mama" (Z853)
  [ 58.5] "desplacament disc intervertebral lumbar" -> "aplastamientos vertebrales" (73313)
  [ 69.0] "historia de violencia de genero" -> "historia de cancer de recto" (Z85048)
  [ 54.5] "golpe contra muro" -> "embolizacion arterial utero" (6825)
  [ 63.2] "acenocumarol" -> "adenoma" (81400)
  [ 65.1] "reaccion aguda al estres" -> "reaccion adaptativa" (F4320)
  [ 69.6] "noduls pulmonars migratorios" -> "nodulos pulmonares" (51889)
  [ 62.5] "nevus" -> "enterovirus" (B341)
  [ 64.0] "ulcus bulbar" -> "ulceras vulva" (N766)
  [ 66.7] "661.43" -> "646.31" (O2623)
  [ 66.7] "aplasia medular" -> "afasia residual" (43811)


## 8. Choose a threshold using validation

Rather than guessing a threshold, we measure accuracy on a held-out portion of the training data. We split training 80/20, use the 80% as the lookup base, and measure how often the predictions on the 20% are correct at different thresholds.

The goal is to find the threshold that maximizes overall accuracy on the validation set. This same threshold will then be applied to the real test predictions.

In [14]:
from sklearn.model_selection import train_test_split

train_split, val_split = train_test_split(train, test_size=0.2, random_state=42)
print(f'Train split: {len(train_split):,} rows')
print(f'Val split:   {len(val_split):,} rows')

Train split: 10,960 rows
Val split:   2,740 rows


In [15]:
# Build the lookup using only the train_split
val_lookup = {}
for lit_norm, group in train_split.groupby('Literal_norm'):
    val_lookup[lit_norm] = group['Code'].value_counts().index[0]

# Apply exact matching to the validation set
val_split = val_split.copy()
val_split['exact_pred'] = val_split['Literal_norm'].map(val_lookup)

# Measure exact match accuracy
val_matched = val_split[val_split['exact_pred'].notna()]
val_exact_correct = (val_matched['exact_pred'] == val_matched['Code']).sum()

print(f'Exact matches in validation: {len(val_matched):,} / {len(val_split):,} '
      f'({len(val_matched)/len(val_split)*100:.1f}%)')
print(f'Of those, correct:           {val_exact_correct:,} '
      f'({val_exact_correct/len(val_matched)*100:.1f}% accuracy on matched)')

Exact matches in validation: 1,474 / 2,740 (53.8%)
Of those, correct:           623 (42.3% accuracy on matched)


In [16]:
# Now apply fuzzy matching to the unmatched validation literals
val_unmatched = val_split[val_split['exact_pred'].isna()]
val_train_literals = list(val_lookup.keys())

print(f'Running fuzzy matching on {len(val_unmatched):,} unmatched validation literals...')

val_fuzzy = []
for idx, row in val_unmatched.iterrows():
    result = process.extractOne(
        row['Literal_norm'],
        val_train_literals,
        scorer=fuzz.ratio,
        score_cutoff=0
    )
    if result is not None:
        val_fuzzy.append({
            'val_idx': idx,
            'matched_code': val_lookup[result[0]],
            'score': result[1],
            'true_code': val_split.loc[idx, 'Code']
        })

val_fuzzy_df = pd.DataFrame(val_fuzzy)
print(f'Got {len(val_fuzzy_df):,} fuzzy results')

Running fuzzy matching on 1,266 unmatched validation literals...
Got 1,266 fuzzy results


In [17]:
# Test different thresholds and find the best overall accuracy
print(f'{"Threshold":>10} {"Fuzzy n":>10} {"Fuzzy ok":>10} {"Fuzzy acc":>11} {"Overall acc":>13}')
print('-' * 60)

results_by_threshold = []
for t in [50, 55, 60, 65, 70, 75, 80, 85, 90, 95]:
    accepted = val_fuzzy_df[val_fuzzy_df.score >= t]
    n_fuzzy = len(accepted)
    n_correct = (accepted['matched_code'] == accepted['true_code']).sum()
    fuzzy_acc = n_correct / n_fuzzy * 100 if n_fuzzy > 0 else 0
    overall_acc = (val_exact_correct + n_correct) / len(val_split) * 100
    results_by_threshold.append((t, n_fuzzy, n_correct, fuzzy_acc, overall_acc))
    print(f'{t:>10} {n_fuzzy:>10,} {n_correct:>10,} {fuzzy_acc:>10.1f}% {overall_acc:>12.1f}%')

# Find the best threshold
best = max(results_by_threshold, key=lambda x: x[4])
print(f'\nBest threshold: {best[0]} (overall accuracy {best[4]:.1f}%)')

 Threshold    Fuzzy n   Fuzzy ok   Fuzzy acc   Overall acc
------------------------------------------------------------
        50      1,266        417       32.9%         38.0%
        55      1,261        417       33.1%         38.0%
        60      1,245        417       33.5%         38.0%
        65      1,186        414       34.9%         37.8%
        70      1,069        406       38.0%         37.6%
        75        949        387       40.8%         36.9%
        80        795        368       46.3%         36.2%
        85        604        331       54.8%         34.8%
        90        454        280       61.7%         33.0%
        95        218        140       64.2%         27.8%

Best threshold: 50 (overall accuracy 38.0%)


## 9. Generate Step 1 predictions on the real test set

We use the best threshold found from validation. Some literals will remain unresolved after Step 1 — those will be passed to Step 2 (semantic retrieval).

In [18]:
# Use the threshold we found in validation. 
# If you ran the cell above, you can replace this value with the printed best one.
FUZZY_THRESHOLD = 60

# Build the predictions
test['step1_code'] = test['exact_code']
test['step1_method'] = test['exact_code'].apply(lambda x: 'exact' if pd.notna(x) else 'unresolved')

# Fill in fuzzy results for rows where exact failed and fuzzy is above threshold
for _, fz in fuzzy_df.iterrows():
    if fz['fuzzy_score'] >= FUZZY_THRESHOLD:
        test.loc[fz['test_idx'], 'step1_code'] = fz['matched_code']
        test.loc[fz['test_idx'], 'step1_method'] = 'fuzzy'

print(f'Step 1 predictions (threshold = {FUZZY_THRESHOLD}):')
print(test['step1_method'].value_counts())
print()
print(f'Resolved by Step 1: {test["step1_code"].notna().sum():,} '
      f'({test["step1_code"].notna().mean()*100:.1f}%)')
print(f'Left for Step 2:    {test["step1_code"].isna().sum():,} '
      f'({test["step1_code"].isna().mean()*100:.1f}%)')

Step 1 predictions (threshold = 60):
step1_method
exact         3812
fuzzy         2819
unresolved      36
Name: count, dtype: int64

Resolved by Step 1: 6,631 (99.5%)
Left for Step 2:    36 (0.5%)


## 10. Save the output for Step 2

Step 2 needs to know which literals were resolved by Step 1 and which still need attention. We save everything in one file.

In [ ]:
output = test[['id', 'Literal', 'Literal_clean', 'Literal_norm', 'step1_code', 'step1_method']].copy()
output.to_csv('step1_predictions.csv', index=False)

print('Saved step1_predictions.csv')
print(f'\nSample of the output:')
output.sample(8, random_state=5)

Saved step1_predictions.csv

Sample of the output:


,id,Literal,Literal_clean,Literal_norm,step1_code,step1_method
5506,5507,Macrosomía fetal,Macrosomía fetal,macrosomia fetal,O3663X0,exact
5286,5287,carcinoma infiltrante,carcinoma infiltrante,carcinoma infiltrante,85003,exact
2851,2852,obstrucción carotídeo y vertebral derecho,obstrucción carotídeo y vertebral derecho,obstruccion carotideo y vertebral derecho,J449,fuzzy
6628,6629,Limfopènia,Limfopènia,limfopenia,D72810,exact
4236,4237,Hipertensión Insuficiencia cardíaca,Hipertensión Insuficiencia cardíaca,hipertension insuficiencia cardiaca,I110,exact
4951,4952,LINFADENECTOMÍA PÉLVICA,LINFADENECTOMÍA PÉLVICA,linfadenectomia pelvica,4053,exact
250,251,lumbociatica,lumbociatica,lumbociatica,7243,exact
1246,1247,Grupo sanguineo: 0+,Grupo sanguineo: 0+,grupo sanguineo: 0+,Z6740,exact


: 